<a href="https://colab.research.google.com/github/chrislouis86/Weights-Biases/blob/main/03_schemavalidation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!git clone https://github.com/wandb/edu.git
!cp -r edu/ml-observability/dataval .
# Verify the directory exists
import os
if os.path.exists('dataval'):
    print('dataval module successfully copied.')

Cloning into 'edu'...
remote: Enumerating objects: 4877, done.
remote: Counting objects: 100% (1321/1321), done.
remote: Compressing objects: 100% (439/439), done.
remote: Total 4877 (delta 1102), reused 907 (delta 880), pack-reused 3556 (from 3)
Receiving objects: 100% (4877/4877), 41.98 MiB | 26.47 MiB/s, done.
Resolving deltas: 100% (2675/2675), done.
cp: cannot stat 'edu/ml-observability/dataval': No such file or directory


![](https://raw.githubusercontent.com/wandb/wandb/508982e50e82c54cbf0dd464a9959fee0e1740ad/.github/wb-logo-lightbg.png)
<!--- @wandbcode{dataval-course-03} -->

# Schema Validation

In this notebook, we will implement TFX's schema validation to see if any of the corruptions in the previous notebook were detected. We'll log the results of schema validation to wandb.

You can set up wandb alerts here: https://docs.wandb.ai/guides/runs/alert

I use Modal because TFDV doesn't run on Mac M1s. You can create a free account on Modal here: https://modal.com/signup -- it comes with $10/month credits, which should be plenty more than enough to run the notebooks in this course. Once you have created an account, follow the "Getting Started" instructions on the homepage:

* Run `pip install modal-client` (also included in `requirements.txt` in this repo)
* Run `modal token new`, which will open a browser window and authenticate you with your account

Then you should be able to run this notebook!

In [29]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import wandb
from wandb import AlertLevel

# 1. Fix Modal and internal dependencies installation
!pip uninstall -y modal-client
!pip install modal catboost

# 2. Fix 'dataval' missing error by finding the correct path in the repo
if not os.path.exists('dataval'):
    if not os.path.exists('edu'):
        !git clone https://github.com/wandb/edu.git
    # Locate the dataval directory and copy it to root
    dataval_path = !find edu -name "dataval" -type d | head -n 1
    if dataval_path:
        !cp -r {dataval_path[0]} .
        print(f"Copied dataval from {dataval_path[0]}")

# 3. Import libraries
import modal
from dataval.dataset import WeatherDataset
from dataval.train import CatBoostTrainer

In [37]:
# Download the dataset required by the notebook
import os
if not os.path.exists('canonical-paritioned-dataset'):
    print('Downloading dataset...')
    # The dataset is located in the ml-observability folder of the wandb/edu repo
    !wget -q https://github.com/wandb/edu/raw/main/ml-observability/canonical-paritioned-dataset.zip
    !unzip -q canonical-paritioned-dataset.zip
    # Ensure the folder is in the current working directory
    if not os.path.exists('canonical-paritioned-dataset') and os.path.exists('ml-observability/canonical-paritioned-dataset'):
        !mv ml-observability/canonical-paritioned-dataset .
    print('Dataset extracted successfully.')

unzip:  cannot find or open canonical-paritioned-dataset.zip, canonical-paritioned-dataset.zip.zip or canonical-paritioned-dataset.zip.ZIP.
Dataset extracted successfully.


In [33]:
# Download the dataset required by the notebook
import os
if not os.path.exists('canonical-paritioned-dataset'):
    print("Downloading dataset...")
    !wget -q https://github.com/wandb/edu/raw/main/ml-dataval-course/canonical-paritioned-dataset.zip
    !unzip -q canonical-paritioned-dataset.zip
    print("Dataset extracted successfully.")

unzip:  cannot find or open canonical-paritioned-dataset.zip, canonical-paritioned-dataset.zip.zip or canonical-paritioned-dataset.zip.ZIP.
Dataset extracted successfully.


In [31]:
image = (
    modal.Image.debian_slim()
    .pip_install(["tensorflow-data-validation", "tensorflow_metadata", "protobuf==3.20.0"])
)
# Changed modal.Stub to modal.App per the error message recommendation
app = modal.App("tfdv-tutorial", image=image)
stub = app # Maintaining 'stub' variable name to avoid breaking downstream cells

In [ ]:
# Load dataset
import os
import shutil

dataset_dir = "canonical-paritioned-dataset"

# The repo includes a download script for the dataset
if not os.path.exists(dataset_dir) or not os.listdir(dataset_dir):
    print("Dataset missing. Running download script from repo...")
    # Navigate to the course directory to run the download script
    !cd edu/ml-dataval-course && bash download.sh
    # Move the downloaded folder to the root content directory
    if os.path.exists("edu/ml-dataval-course/canonical-paritioned-dataset"):
        !mv edu/ml-dataval-course/canonical-paritioned-dataset .

ds = WeatherDataset(os.path.abspath(dataset_dir), sample_frac=0.2)

Dataset missing. Running download script from repo...
--2026-09-14 15:01:41--  https://storage.yandexcloud.net/yandex-research/shifts/weather/canonical-partitioned-dataset.tar
Resolving storage.yandexcloud.net (storage.yandexcloud.net)... 213.180.193.243, 2a02:6b8::1d9
Connecting to storage.yandexcloud.net (storage.yandexcloud.net)|213.180.193.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7472220160 (7.0G) [application/x-tar]
Saving to: ‘canonical-partitioned-dataset.tar’

canonical-partition 100%[===================>]   6.96G  23.2MB/s    in 5m 9s   

2026-09-14 15:06:51 (23.0 MB/s) - ‘canonical-partitioned-dataset.tar’ saved [7472220160/7472220160]

canonical-paritioned-dataset/
canonical-paritioned-dataset/LICENSE.md
canonical-paritioned-dataset/shifts_canonical_eval_out.csv
canonical-paritioned-dataset/shifts_canonical_dev_in.csv
canonical-paritioned-dataset/shifts_canonical_train.csv
canonical-paritioned-dataset/shifts_canonical_eval_in.csv
canonica

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
# Initialize ds if it's not in the namespace (prevents NameError after long downloads)
if 'ds' not in globals():
    from dataval.dataset import WeatherDataset
    import os
    dataset_dir = os.path.abspath("canonical-paritioned-dataset")
    ds = WeatherDataset(dataset_dir, sample_frac=0.2)

train_df = ds.load(ds.get_partition_keys()[0])
test_df = ds.load(ds.get_partition_keys()[1])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### Modal API Updates Summary

1. **Repository Setup**:
```bash
!git clone https://github.com/wandb/edu.git
!cp -r edu/ml-dataval-course/dataval .
```

2. **Modal App Initialization**:
Update `modal.Stub` to `modal.App`:
```python
app = modal.App("tfdv-tutorial", image=image)
stub = app # Alias for compatibility
```

3. **Function Remote Calls**:
Use `.remote()` instead of `.call()` if using newer Modal features:
```python
anomalies = find_anomalies.remote(X_train, X_test)
```

In [18]:
train_df

NameError: name 'train_df' is not defined

## Infer schema and check test data for errors

From the train dataframe, we create a schema using TFDV. Then we use this schema to find anomalies in the test data. We apply this to the original dataframes first.

In [ ]:
@stub.function()
def find_anomalies(train_df, test_df):
    import tensorflow_data_validation as tfdv
    from google.protobuf.json_format import MessageToDict

    train_stats =  tfdv.generate_statistics_from_dataframe(train_df)
    schema = tfdv.infer_schema(statistics=train_stats)
    test_stats = tfdv.generate_statistics_from_dataframe(test_df)

    anomalies = tfdv.validate_statistics(statistics=test_stats, schema=schema)
    anomalies_df = tfdv.utils.display_util.get_anomalies_dataframe(anomalies)
    # return MessageToDict(anomalies)
    return anomalies_df

In [ ]:
with stub.run():
    X_train, _ = ds.split_feature_label(train_df)
    X_test, _ = ds.split_feature_label(test_df)
    anomalies = find_anomalies.call(X_train, X_test)

⠹ Running (1/1 containers active)... View app at https://modal.com/apps/ap-G9gTqWePyph4JVOhAcuftw

✓ App completed.

In [ ]:
anomalies

,Anomaly short description,Anomaly long description
Feature name,,


Seems like the raw data did not have any anomalies!

## Iterate through corruptions

See if tfdv detects any anomalies, for all the corruptions we had in our previous notebook.

In [ ]:
X_train, _ = ds.split_feature_label(train_df)
corruption_anomalies = {}

for corruption_name, corruption_res in ds.iterate_corruptions(test_df, "cmc", corruption_rate=0.05):
    corrupted_test_df, corrupted_columns = corruption_res
    corrupted_X_test, _ = ds.split_feature_label(corrupted_test_df)
    with stub.run():
        corruption_anomalies[corruption_name] = find_anomalies.call(X_train, corrupted_X_test)

⠏ Running (1/1 containers active)... View app at https://modal.com/apps/ap-2RwOCNeA8im51cbPI3zIe0

✓ App completed.

In [ ]:
# Send wandb alerts

run = wandb.init(project="ml-dataval-tutorial", tags=["TFDV-schema"])

for corruption_name, anomalies in corruption_anomalies.items():
    if len(anomalies) > 0:
        table = wandb.Table(dataframe=anomalies)
        wandb.log({corruption_name: table})

        wandb.alert(
            title=f"Errors detected in {corruption_name} experiment",
            text = f"Found {len(anomalies)} anomalies",
            level=AlertLevel.WARN,
        )

wandb.finish()

wandb: Currently logged in as: darek. Use `wandb login --relogin` to force relogin


## Takeaways

It looks like only the `corrupt_null` corruption was flagged by schema validation! Maybe other validation techniques might flag them. Nevertheless, all the corruptions that schema validation found were true corruptions, so there isn't a false positive alert problem here.